# exp7 baseline retrain

**Self-contained Colab notebook.** Run the setup cells once (top to bottom), then the experiment. The four shared modules are written to `/content` automatically, so you do NOT need to upload a `shared/` folder. Edit Drive paths only in the `config_paths.py` cell below.

> Runtime: set **Runtime -> Change runtime type -> GPU** for the inference and training experiments.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
print('Drive mounted; /content on path.')

Mounted at /content/drive
Drive mounted; /content on path.


In [2]:
# scikit-image backs metrics.calc_ssim; guarantee it for every notebook
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-image'])
print('scikit-image ready')

scikit-image ready


In [3]:
%%writefile /content/config_paths.py
"""
Central path + constant configuration for all JBHI revision Colab experiments.

Derived verbatim from the author's existing notebooks:
  - COLAB_IEEE_FIGURES.ipynb      (paper baselines + EPHNOGRAM chunks + split)
  - cross_dataset_validation.ipynb (SensSmart-Tech + Trail-4 generator)

Every experiment script imports from this file so paths live in ONE place.
If your Drive layout differs, edit ONLY this file.
"""

import os
import torch

# ============================================================================
# >>> EDIT THIS BLOCK IF YOUR DRIVE MOVES <<<
# Updated 2026-07-14: the dataset is now a SHARED folder mounted at
# /content/drive/MyDrive/JBHI-Revision/October Thesis Dataset. If it moves again,
# change ONLY DRIVE_ROOT and OCT_DIR. NOTE: DATA is READ from OCT_DIR; everything
# the code WRITES (retrained checkpoints, outputs) goes under your own Drive
# (OUT_ROOT), so a read-only shared dataset still works.
# ============================================================================
DRIVE_ROOT = "/content/drive/MyDrive"
# Shared dataset root: contains "Refine-Chunked", "Best-epoch-trail-4", etc. (read-only OK).
OCT_DIR    = f"{DRIVE_ROOT}/JBHI-Revision/October Thesis Dataset"

# ----------------------------------------------------------------------------
# EPHNOGRAM data (training-domain test set = 453 segments, 3 subjects)
# .npz keys per chunk: 'ecg', 'pcg', 'subject_id', 'scenario'
#   ecg, pcg : float32 arrays, 64000 samples (8 s @ 8 kHz), RAW (not normalized)
#   subject_id : e.g. 'S009', 'S019', 'S023'
#   scenario   : free-text, e.g. 'rest', 'exercise bicycle stress test'
# ----------------------------------------------------------------------------
EPH_CHUNKS_DIR = f"{OCT_DIR}/Refine-Chunked"
# Subject split. keys: train/val/test_subjects. IF THIS FILE IS ABSENT the
# notebooks fall back to an arbitrary split, which gives the WRONG test set
# (the 2026-07-10 run hit this: 22 segments / 4 subjects instead of the paper's
# 453 segments / 3 subjects). Upload the real split JSON, or point this at it.
SPLIT_FILE     = f"{EPH_CHUNKS_DIR}/train_val_test_split.json"

# ----------------------------------------------------------------------------
# Model checkpoints  --  CONFIRM ALL FOUR EXIST IN THE NEW ACCOUNT
#
#   "Proposed" MUST be the Trail-4 generator (polarity fix, BatchNorm output),
#   i.e. trail4_best.pt or trail4_generator.pth. A "trial2_*" / Trail-2 file is
#   the WRONG (Tanh) model: it loads only partially under strict=False and
#   produces a near-flat, invalid reconstruction (see the 2026-07-10 run).
#
#   The three baselines lived under a SEPARATE top-level folder
#   (Paper1_ECG_Reconstruction) in the old account. Confirm they were migrated;
#   if they now sit under OCT_DIR, repoint them below.
# ----------------------------------------------------------------------------
CKPT = {
    # Baselines are RETRAINED by exp7 and SAVED to your own writable Drive
    # (DRIVE_ROOT/JBHI_Revision/baseline_checkpoints == OUT_ROOT/baseline_checkpoints),
    # so a read-only shared dataset is fine. Same paths are used to load them later.
    "CNN":      f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/CNN_EncoderDecoder/best_model.pt",
    "UNet":     f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/UNet_Baseline/best_model.pt",
    "WGAN-GP":  f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/WGAN_GP/best_model.pt",
    "Proposed": f"{OCT_DIR}/Best-epoch-trail-4/trail4_best.pt",   # read from the shared dataset
}
CKPT_PROPOSED_ALT = f"{OCT_DIR}/trail4_generator.pth"   # fallback copy (state_dict)

# ----------------------------------------------------------------------------
# SensSmart-Tech cross-dataset (32 subjects, 338 records @ 1 kHz -> 8 kHz)
#   WFDB pairs: {base}_ecg.hea/.dat and {base}_pcg.hea/.dat ; ECG lead II
# ----------------------------------------------------------------------------
SENSSMART_WFDB  = f"{OCT_DIR}/senssmarttech-database/WFDB"
SENSSMART_PREP  = f"{OCT_DIR}/milestone2_preprocessed"     # cached .npz (keys ecg/pcg)

# ----------------------------------------------------------------------------
# Output root for ALL revision artifacts (created on demand)
#   Each experiment writes to a named sub-folder here.
# ----------------------------------------------------------------------------
OUT_ROOT = f"{DRIVE_ROOT}/JBHI_Revision"

def out_dir(name):
    """Return (and create) an output sub-folder under OUT_ROOT."""
    p = os.path.join(OUT_ROOT, name)
    os.makedirs(p, exist_ok=True)
    return p

# ----------------------------------------------------------------------------
# Signal + inference constants
# ----------------------------------------------------------------------------
FS          = 8000        # sampling rate (Hz)
SEG_LEN     = 64000       # samples per segment (8 s)
NUM_SCENARIOS = 3         # rest / exercise / recovery one-hot
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# R-peak detector settings (identical to metrics.ipynb; keep fixed for reproducibility)
RPEAK_DISTANCE   = 400    # min inter-peak samples = 50 ms @ 8 kHz
RPEAK_PROMINENCE = 0.5

# SensSmart preprocessing bands (from cross_dataset_validation.ipynb)
ECG_BAND = (0.5, 40.0)
PCG_BAND = (25.0, 400.0)
FILTER_ORDER = 4

# Scenario one-hots
SCEN_REST     = [1.0, 0.0, 0.0]
SCEN_EXERCISE = [0.0, 1.0, 0.0]
SCEN_RECOVERY = [0.0, 0.0, 1.0]


def scenario_to_index(scenario_str):
    """Map EPHNOGRAM free-text scenario -> class index {0,1,2}.

    Matches the mapping used in COLAB_IEEE_FIGURES.ipynb.
    """
    s = str(scenario_str).lower()
    if "rest" in s or "laying" in s:
        return 0
    if "exercise" in s or "bicycle" in s or "treadmill" in s or "running" in s:
        return 1
    return 2

Writing /content/config_paths.py


In [4]:
%%writefile /content/models.py
"""
Model architectures for the JBHI revision experiments.

Reproduced VERBATIM from the author's COLAB_IEEE_FIGURES.ipynb (paper Table II
baselines) and models.ipynb (ablation trails), so the checkpoints in
config_paths.CKPT load cleanly.

Which class loads which checkpoint
----------------------------------
    CKPT['CNN']      -> CNNEncoderDecoder()        # no scenario, forward(x)
    CKPT['UNet']     -> UNetGenerator()            # pix2pix, forward(x)
    CKPT['WGAN-GP']  -> UNetGenerator()            # same generator as U-Net
    CKPT['Proposed'] -> ImprovedGeneratorNoTanh()  # scenario, forward(x, s_onehot)

The proposed model's forward takes a scenario one-hot; the three baselines take
only the PCG. Every inference script must branch on this (helper `run_generator`
below does it for you).

For the R1.4 component ablation, use `ConfigurableGenerator`, which exposes the
three grouped factors as switches: use_attention, use_dilated, and
output_activation in {'batchnorm','tanh','linear'}.

NOTE ON PARAM COUNTS: the submitted paper's numbers (CNN 9.8M, U-Net 13.6M,
Proposed 14.0M) do NOT match these classes (which give ~larger counts). Always
report count_params(model_loaded_from_checkpoint) computed here -- see R2.2/R2.5.
"""

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ===========================================================================
# Shared bottleneck modules
# ===========================================================================
class MultiHeadSelfAttention(nn.Module):
    """8-head self-attention over the temporal axis at the bottleneck."""
    def __init__(self, channels, num_heads=8, dropout=0.1):
        super().__init__()
        assert channels % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1, bias=False)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, C, T = x.shape
        qkv = self.qkv(x).reshape(B, 3, self.num_heads, self.head_dim, T)
        qkv = qkv.permute(1, 0, 2, 4, 3)          # (3, B, heads, T, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]
        # Memory-efficient / flash attention: O(T) memory instead of materializing
        # the T x T score matrix, which is what OOM-ed the attention variants on a
        # 14.5 GB T4. It is mathematically identical to softmax(QK^T/sqrt(d))V (SDPA
        # scales by 1/sqrt(head_dim) = self.scale), so checkpoints trained with the
        # old manual path load and evaluate unchanged.
        p = self.dropout.p if self.training else 0.0
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=p)
        out = out.transpose(2, 3).reshape(B, C, T)
        return self.proj(out) + x


class MultiScaleDilatedBlock(nn.Module):
    """Parallel dilated convs at rates 1, 2, 4 with a 1x1 fusion + residual."""
    def __init__(self, channels):
        super().__init__()
        c1 = channels // 3 + (1 if channels % 3 >= 1 else 0)
        c2 = channels // 3 + (1 if channels % 3 >= 2 else 0)
        c3 = channels // 3
        self.conv_d1 = nn.Conv1d(channels, c1, 3, padding=1, dilation=1)
        self.conv_d2 = nn.Conv1d(channels, c2, 3, padding=2, dilation=2)
        self.conv_d4 = nn.Conv1d(channels, c3, 3, padding=4, dilation=4)
        self.fusion = nn.Sequential(
            nn.Conv1d(channels, channels, 1),
            nn.BatchNorm1d(channels), nn.ReLU(inplace=True))

    def forward(self, x):
        cat = torch.cat([self.conv_d1(x), self.conv_d2(x), self.conv_d4(x)], dim=1)
        return self.fusion(cat) + x


class EnhancedResidualBlock(nn.Module):
    """Residual block with optional MHSA and MSDC (Trails 2-4)."""
    def __init__(self, channels, use_attention=False, use_dilated=True):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(channels)
        self.use_dilated = use_dilated
        if use_dilated:
            self.dilated_block = MultiScaleDilatedBlock(channels)
        self.conv2 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)
        self.use_attention = use_attention
        if use_attention:
            self.attention = MultiHeadSelfAttention(channels, num_heads=8)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        if self.use_dilated:
            out = self.dilated_block(out)
        out = self.bn2(self.conv2(out))
        if self.use_attention:
            out = self.attention(out)
        return self.relu(out + residual)


# ===========================================================================
# Paper Table II baseline: CNN Encoder-Decoder (8-level, no skips, no scenario)
# ===========================================================================
class ConvBlock(nn.Module):
    def __init__(self, i, o, k=4, s=2, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(i, o, k, s, p, bias=False),
            nn.BatchNorm1d(o), nn.LeakyReLU(0.2, inplace=True))
    def forward(self, x): return self.block(x)


class DeconvBlock(nn.Module):
    def __init__(self, i, o, k=4, s=2, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose1d(i, o, k, s, p, bias=False),
            nn.BatchNorm1d(o), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)


class CNNEncoderDecoder(nn.Module):
    """Paper CNN baseline -> loads CKPT['CNN']. forward(x)."""
    def __init__(self, in_channels=1, out_channels=1, base_filters=64):
        super().__init__()
        bf = base_filters
        self.enc1 = ConvBlock(in_channels, bf)
        self.enc2 = ConvBlock(bf, bf * 2)
        self.enc3 = ConvBlock(bf * 2, bf * 4)
        self.enc4 = ConvBlock(bf * 4, bf * 8)
        self.enc5 = ConvBlock(bf * 8, bf * 8)
        self.enc6 = ConvBlock(bf * 8, bf * 8)
        self.enc7 = ConvBlock(bf * 8, bf * 8)
        self.enc8 = ConvBlock(bf * 8, bf * 8)
        self.dec1 = DeconvBlock(bf * 8, bf * 8)
        self.dec2 = DeconvBlock(bf * 8, bf * 8)
        self.dec3 = DeconvBlock(bf * 8, bf * 8)
        self.dec4 = DeconvBlock(bf * 8, bf * 8)
        self.dec5 = DeconvBlock(bf * 8, bf * 4)
        self.dec6 = DeconvBlock(bf * 4, bf * 2)
        self.dec7 = DeconvBlock(bf * 2, bf)
        self.final = nn.ConvTranspose1d(bf, out_channels, 4, 2, 1)

    def forward(self, x):
        n = x.size(2)
        e = self.enc1(x); e = self.enc2(e); e = self.enc3(e); e = self.enc4(e)
        e = self.enc5(e); e = self.enc6(e); e = self.enc7(e); e = self.enc8(e)
        d = self.dec1(e); d = self.dec2(d); d = self.dec3(d); d = self.dec4(d)
        d = self.dec5(d); d = self.dec6(d); d = self.dec7(d)
        out = self.final(d)
        if out.size(2) != n:
            diff = n - out.size(2)
            out = F.pad(out, (diff // 2, diff - diff // 2))
        return out


# ===========================================================================
# Paper Table II baseline: pix2pix U-Net (shared by U-Net and WGAN-GP)
# ===========================================================================
class UNetBlock(nn.Module):
    def __init__(self, i, o, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv1d(i, o, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.BatchNorm1d(o))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)


class UNetUpBlock(nn.Module):
    def __init__(self, i, o, dropout=0.0):
        super().__init__()
        layers = [nn.ConvTranspose1d(i, o, 4, 2, 1, bias=False),
                  nn.BatchNorm1d(o), nn.ReLU(inplace=True)]
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x, skip):
        x = self.block(x)
        if x.size(2) != skip.size(2):
            diff = skip.size(2) - x.size(2)
            x = F.pad(x, (diff // 2, diff - diff // 2))
        return torch.cat([x, skip], dim=1)


class UNetGenerator(nn.Module):
    """Paper U-Net / WGAN-GP generator -> loads CKPT['UNet'] or CKPT['WGAN-GP']. forward(x)."""
    def __init__(self, in_channels=1, out_channels=1, features=64):
        super().__init__()
        f = features
        self.e1 = UNetBlock(in_channels, f, normalize=False)
        self.e2 = UNetBlock(f, f * 2); self.e3 = UNetBlock(f * 2, f * 4)
        self.e4 = UNetBlock(f * 4, f * 8); self.e5 = UNetBlock(f * 8, f * 8)
        self.e6 = UNetBlock(f * 8, f * 8); self.e7 = UNetBlock(f * 8, f * 8)
        self.e8 = UNetBlock(f * 8, f * 8, normalize=False)
        self.d1 = UNetUpBlock(f * 8, f * 8, dropout=0.5)
        self.d2 = UNetUpBlock(f * 16, f * 8, dropout=0.5)
        self.d3 = UNetUpBlock(f * 16, f * 8, dropout=0.5)
        self.d4 = UNetUpBlock(f * 16, f * 8)
        self.d5 = UNetUpBlock(f * 16, f * 4); self.d6 = UNetUpBlock(f * 8, f * 2)
        self.d7 = UNetUpBlock(f * 4, f)
        self.final = nn.ConvTranspose1d(f * 2, out_channels, 4, 2, 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(e1); e3 = self.e3(e2); e4 = self.e4(e3)
        e5 = self.e5(e4); e6 = self.e6(e5); e7 = self.e7(e6); e8 = self.e8(e7)
        d = self.d1(e8, e7); d = self.d2(d, e6); d = self.d3(d, e5); d = self.d4(d, e4)
        d = self.d5(d, e3); d = self.d6(d, e2); d = self.d7(d, e1)
        out = self.final(d)
        if out.size(2) != x.size(2):
            diff = x.size(2) - out.size(2)
            out = F.pad(out, (diff // 2, diff - diff // 2))
        return out


# ===========================================================================
# Proposed generator (Trail-4): U-Net + MHSA + MSDC + scenario, BatchNorm output
# ===========================================================================
class ImprovedGeneratorNoTanh(nn.Module):
    """Proposed model -> loads CKPT['Proposed']. forward(x, scenario_onehot)."""
    def __init__(self, signal_length=64000, base_channels=64, num_scenarios=3):
        super().__init__()
        bc = base_channels
        self.scenario_embed = nn.Linear(num_scenarios, bc)
        self.enc1 = nn.Sequential(nn.Conv1d(1, bc, 7, 2, 3), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv1d(bc, bc*2, 5, 2, 2), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv1d(bc*2, bc*4, 5, 2, 2), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.enc4 = nn.Sequential(nn.Conv1d(bc*4, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.enc5 = nn.Sequential(nn.Conv1d(bc*8, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.bottleneck = nn.Sequential(
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True))
        self.dec5 = nn.Sequential(nn.ConvTranspose1d(bc*8, bc*8, 4, 2, 1), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.dec4 = nn.Sequential(nn.Conv1d(bc*16, bc*8, 1), nn.ConvTranspose1d(bc*8, bc*4, 4, 2, 1), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.dec3 = nn.Sequential(nn.Conv1d(bc*8, bc*4, 1), nn.ConvTranspose1d(bc*4, bc*2, 4, 2, 1), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.Conv1d(bc*4, bc*2, 1), nn.ConvTranspose1d(bc*2, bc, 4, 2, 1), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.dec1 = nn.Sequential(nn.Conv1d(bc*2, bc, 1), nn.ConvTranspose1d(bc, 1, 4, 2, 1), nn.BatchNorm1d(1))

    def forward(self, x, scenario_onehot):
        s = self.scenario_embed(scenario_onehot).unsqueeze(2)
        e1 = self.enc1(x) + s
        e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3); e5 = self.enc5(e4)
        b = self.bottleneck(e5)
        d5 = torch.cat([self.dec5(b), e4], dim=1)
        d4 = torch.cat([self.dec4(d5), e3], dim=1)
        d3 = torch.cat([self.dec3(d4), e2], dim=1)
        d2 = torch.cat([self.dec2(d3), e1], dim=1)
        return self.dec1(d2)


# ===========================================================================
# Configurable generator for the R1.4 component ablation
# ===========================================================================
class ConfigurableGenerator(nn.Module):
    """Ablation generator: toggle each grouped factor independently.

    use_attention : add MHSA in the 4 bottleneck blocks
    use_dilated   : add MSDC in the 4 bottleneck blocks
    output_activation : 'batchnorm' (proposed) | 'tanh' | 'linear'
    use_scenario  : condition on the scenario one-hot (keep True to match proposed)

    Ablation rows the reviewer asked for map to:
        U-Net only          -> use_attention=False, use_dilated=False
        U-Net + MSDC        -> use_attention=False, use_dilated=True
        U-Net + MHSA        -> use_attention=True,  use_dilated=False
        U-Net + both        -> use_attention=True,  use_dilated=True
        Proposed + Tanh     -> both True, output_activation='tanh'
        Proposed + linear   -> both True, output_activation='linear'
        Proposed (BatchNorm)-> both True, output_activation='batchnorm'
    Polarity-on/off is a DATA-loader switch; adversarial-on/off is a TRAINING
    switch -- neither is an architecture change, so control them in the trainer.
    """
    def __init__(self, base_channels=64, num_scenarios=3,
                 use_attention=True, use_dilated=True,
                 output_activation="batchnorm", use_scenario=True):
        super().__init__()
        bc = base_channels
        self.use_scenario = use_scenario
        if use_scenario:
            self.scenario_embed = nn.Linear(num_scenarios, bc)
        self.enc1 = nn.Sequential(nn.Conv1d(1, bc, 7, 2, 3), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv1d(bc, bc*2, 5, 2, 2), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv1d(bc*2, bc*4, 5, 2, 2), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.enc4 = nn.Sequential(nn.Conv1d(bc*4, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.enc5 = nn.Sequential(nn.Conv1d(bc*8, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.bottleneck = nn.Sequential(*[
            EnhancedResidualBlock(bc*8, use_attention=use_attention, use_dilated=use_dilated)
            for _ in range(4)])
        self.dec5 = nn.Sequential(nn.ConvTranspose1d(bc*8, bc*8, 4, 2, 1), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.dec4 = nn.Sequential(nn.Conv1d(bc*16, bc*8, 1), nn.ConvTranspose1d(bc*8, bc*4, 4, 2, 1), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.dec3 = nn.Sequential(nn.Conv1d(bc*8, bc*4, 1), nn.ConvTranspose1d(bc*4, bc*2, 4, 2, 1), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.Conv1d(bc*4, bc*2, 1), nn.ConvTranspose1d(bc*2, bc, 4, 2, 1), nn.BatchNorm1d(bc), nn.ReLU(True))
        head = [nn.Conv1d(bc*2, bc, 1), nn.ConvTranspose1d(bc, 1, 4, 2, 1)]
        if output_activation == "batchnorm":
            head.append(nn.BatchNorm1d(1))
        elif output_activation == "tanh":
            head.append(nn.Tanh())
        elif output_activation == "linear":
            pass
        else:
            raise ValueError(f"unknown output_activation {output_activation}")
        self.dec1 = nn.Sequential(*head)

    def forward(self, x, scenario_onehot=None):
        e1 = self.enc1(x)
        if self.use_scenario and scenario_onehot is not None:
            e1 = e1 + self.scenario_embed(scenario_onehot).unsqueeze(2)
        e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3); e5 = self.enc5(e4)
        b = self.bottleneck(e5)
        d5 = torch.cat([self.dec5(b), e4], dim=1)
        d4 = torch.cat([self.dec4(d5), e3], dim=1)
        d3 = torch.cat([self.dec3(d4), e2], dim=1)
        d2 = torch.cat([self.dec2(d3), e1], dim=1)
        return self.dec1(d2)


# ===========================================================================
# Discriminator (for R1.4 adversarial retraining)
# ===========================================================================
class SpectralNormDiscriminator(nn.Module):
    """Spectrally-normalized PatchGAN critic with scenario conditioning."""
    def __init__(self, base_channels=64, num_scenarios=3):
        super().__init__()
        bc = base_channels
        self.scenario_embed = nn.Linear(num_scenarios, bc)
        sn = nn.utils.spectral_norm
        self.conv1 = sn(nn.Conv1d(2, bc, 7, 2, 3))
        self.conv2 = sn(nn.Conv1d(bc, bc*2, 5, 2, 2)); self.bn2 = nn.BatchNorm1d(bc*2)
        self.conv3 = sn(nn.Conv1d(bc*2, bc*4, 5, 2, 2)); self.bn3 = nn.BatchNorm1d(bc*4)
        self.conv4 = sn(nn.Conv1d(bc*4, bc*8, 5, 2, 2)); self.bn4 = nn.BatchNorm1d(bc*8)
        self.conv5 = sn(nn.Conv1d(bc*8, bc*8, 5, 2, 2)); self.bn5 = nn.BatchNorm1d(bc*8)
        self.pool = nn.AdaptiveAvgPool1d(1); self.fc = nn.Linear(bc*8, 1)
        self.lrelu = nn.LeakyReLU(0.2, True)

    def forward(self, pcg, ecg, scenario_onehot):
        x = torch.cat([pcg, ecg], dim=1)
        s = self.scenario_embed(scenario_onehot).unsqueeze(2)
        x = self.lrelu(self.conv1(x)) + s
        x = self.lrelu(self.bn2(self.conv2(x)))
        x = self.lrelu(self.bn3(self.conv3(x)))
        x = self.lrelu(self.bn4(self.conv4(x)))
        x = self.lrelu(self.bn5(self.conv5(x)))
        return torch.sigmoid(self.fc(self.pool(x).squeeze(2)))


# ===========================================================================
# WGAN-GP critic (for retraining the WGAN-GP baseline; extracted verbatim from
# the original 05_GAN_WGAN_GP notebook). No BatchNorm, no sigmoid -> a valid
# WGAN critic. The generator is the plain UNetGenerator above.
# ===========================================================================
class WGANCritic(nn.Module):
    def __init__(self, in_channels=2, features=64):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(in_channels, features, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features, features * 2, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 2, features * 4, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 4, features * 8, 4, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 8, 1, 4, 1, 1),
        )

    def forward(self, x, y):
        return self.model(torch.cat([x, y], dim=1))


def compute_gradient_penalty(critic, pcg, real_ecg, fake_ecg, device):
    """WGAN-GP gradient penalty. Interpolates the ECG only; PCG conditions."""
    bs = real_ecg.size(0)
    alpha = torch.rand(bs, 1, 1, device=device)
    interp = (alpha * real_ecg + (1 - alpha) * fake_ecg).requires_grad_(True)
    c_interp = critic(pcg, interp)
    grads = torch.autograd.grad(
        outputs=c_interp, inputs=interp,
        grad_outputs=torch.ones_like(c_interp),
        create_graph=True, retain_graph=True)[0]
    grads = grads.view(bs, -1)
    return ((grads.norm(2, dim=1) - 1) ** 2).mean()


# ===========================================================================
# Loading + utility helpers
# ===========================================================================
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def load_model(model, checkpoint_path, device="cpu", strict=False):
    """Load a checkpoint into `model`, handling common save formats."""
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict):
        for key in ("generator_state_dict", "model_state_dict", "state_dict"):
            if key in ckpt:
                ckpt = ckpt[key]; break
    state = {k.replace("module.", ""): v for k, v in ckpt.items()} \
            if isinstance(ckpt, dict) else ckpt
    result = model.load_state_dict(state, strict=strict)
    # Loud guard: a large key mismatch means the checkpoint does not fit this
    # architecture (e.g. a Trail-2/trial2 file loaded into the Trail-4 model).
    n_model = max(len(model.state_dict()), 1)
    n_missing, n_unexpected = len(result.missing_keys), len(result.unexpected_keys)
    tag = os.path.basename(checkpoint_path)
    if n_missing or n_unexpected:
        flag = "  <-- LIKELY WRONG CHECKPOINT FOR THIS MODEL" if n_missing > 0.10 * n_model else ""
        print(f"[load_model] {tag}: {n_missing}/{n_model} params missing, "
              f"{n_unexpected} unexpected{flag}")
    else:
        print(f"[load_model] {tag}: exact match ({n_model} tensors).")
    model.to(device).eval()
    return model, result


def build_paper_models(ckpt_dict, device="cpu"):
    """Load all four paper models. Returns {name: model}."""
    models = {}
    models["CNN"], _     = load_model(CNNEncoderDecoder(), ckpt_dict["CNN"], device)
    models["UNet"], _    = load_model(UNetGenerator(), ckpt_dict["UNet"], device)
    models["WGAN-GP"], _ = load_model(UNetGenerator(), ckpt_dict["WGAN-GP"], device)
    models["Proposed"], _= load_model(ImprovedGeneratorNoTanh(), ckpt_dict["Proposed"], device)
    return models


def run_generator(model, pcg_tensor, scenario_tensor, is_proposed):
    """Uniform inference call. `is_proposed` selects the (x, s) signature."""
    with torch.no_grad():
        if is_proposed:
            out = model(pcg_tensor, scenario_tensor)
        else:
            out = model(pcg_tensor)
    return out.squeeze().cpu().numpy()

Writing /content/models.py


In [5]:
%%writefile /content/metrics.py
"""
Metrics for the JBHI revision.

Two families:
  (A) Signal-fidelity metrics  -- identical to the submitted paper's metrics.ipynb
                                  (PSNR, SNR, SSIM, Pearson r, MAE, RMSE,
                                   R-peak AMPLITUDE error, QRS-width error).
  (B) Temporal R-peak DETECTION metrics  -- NEW, added to answer Reviewer 1 #1.
       These detect R-peaks INDEPENDENTLY in the reconstructed ECG and match
       them to ground-truth peaks within a tolerance window, then report
       sensitivity, PPV, F1, FP, FN, timing-error stats, RR-interval error,
       and heart-rate error. This is the metric the paper was missing.

Keep the detector settings fixed (config_paths.RPEAK_*) so every experiment and
the paper agree.
"""

import numpy as np
import scipy.signal as signal
from scipy.stats import pearsonr


# ---------------------------------------------------------------------------
# R-peak detection (single, shared definition)
# ---------------------------------------------------------------------------
def detect_r_peaks(ecg, fs=8000, distance=400, prominence=0.5):
    """scipy find_peaks on an ECG trace. Returns integer sample indices."""
    peaks, _ = signal.find_peaks(np.asarray(ecg).ravel(),
                                 distance=distance, prominence=prominence)
    return peaks


# ---------------------------------------------------------------------------
# (A) Signal-fidelity metrics
# ---------------------------------------------------------------------------
def calc_psnr(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    mse = np.mean((real - fake) ** 2)
    if mse < 1e-10:
        return 100.0
    peak = np.max(np.abs(real))
    return float(20 * np.log10(peak / np.sqrt(mse)))


def calc_snr(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    noise = real - fake
    return float(10 * np.log10(np.sum(real ** 2) / (np.sum(noise ** 2) + 1e-12)))


def calc_correlation(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    try:
        c, _ = pearsonr(real, fake)
        return float(c)
    except Exception:
        return 0.0


def calc_ssim(real, fake):
    from skimage.metrics import structural_similarity as compare_ssim  # lazy: Colab-only dep
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    r = (real - real.min()) / (real.max() - real.min() + 1e-8)
    f = (fake - fake.min()) / (fake.max() - fake.min() + 1e-8)
    return float(compare_ssim(r, f, data_range=1.0))


def calc_mae(real, fake):
    return float(np.mean(np.abs(np.asarray(real).ravel() - np.asarray(fake).ravel())))


def calc_rmse(real, fake):
    d = np.asarray(real).ravel() - np.asarray(fake).ravel()
    return float(np.sqrt(np.mean(d ** 2)))


def calc_r_peak_amplitude_error(real, fake, fs=8000, **kw):
    """Paper's 'R-peak error' -> renamed 'R-peak AMPLITUDE error'.

    Detects R-peaks on the GROUND TRUTH only and averages |y(t_k)-yhat(t_k)|.
    Kept ONLY as a fidelity measure; it is NOT a timing metric (Reviewer 1 #1).
    Returns np.nan when the ground truth has no detectable R-peaks (report the
    valid-N denominator alongside the mean).
    """
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    peaks = detect_r_peaks(real, fs=fs, **kw)
    if len(peaks) == 0:
        return np.nan
    return float(np.mean(np.abs(real[peaks] - fake[peaks])))


def _qrs_width_at_peak(ecg, peak_idx, fs=8000, threshold=0.5, search_ms=25):
    amp = abs(ecg[peak_idx]); thr = threshold * amp
    win = int(search_ms / 1000 * fs)
    q = peak_idx
    for j in range(peak_idx - 1, max(peak_idx - win, 0), -1):
        if abs(ecg[j]) < thr:
            q = j; break
    s = peak_idx
    for j in range(peak_idx + 1, min(peak_idx + win, len(ecg))):
        if abs(ecg[j]) < thr:
            s = j; break
    return (s - q) / fs * 1000.0


def calc_qrs_width_error(real, fake, fs=8000, **kw):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    peaks = detect_r_peaks(real, fs=fs, **kw)
    if len(peaks) == 0:
        return np.nan
    errs = [abs(_qrs_width_at_peak(real, p, fs) - _qrs_width_at_peak(fake, p, fs))
            for p in peaks]
    return float(np.mean(errs))


# ---------------------------------------------------------------------------
# (B) Temporal R-peak DETECTION metrics  (NEW -- Reviewer 1 #1)
# ---------------------------------------------------------------------------
def match_peaks(true_peaks, pred_peaks, tol_samples):
    """Greedy nearest-neighbour matching within a tolerance window.

    Each true peak claims the closest unused predicted peak within tol_samples.
    Returns (matches, TP, FP, FN, signed_timing_errors_samples).
    signed error = pred_index - true_index (positive = reconstructed peak late).
    """
    true_peaks = list(np.asarray(true_peaks).astype(int))
    pred_peaks = list(np.asarray(pred_peaks).astype(int))
    used, matches, terr = set(), [], []
    for tp in true_peaks:
        cands = [(abs(pp - tp), pp) for pp in pred_peaks
                 if pp not in used and abs(pp - tp) <= tol_samples]
        if cands:
            _, pp = min(cands)
            used.add(pp); matches.append((tp, pp)); terr.append(pp - tp)
    TP = len(matches)
    FN = len(true_peaks) - TP
    FP = len(pred_peaks) - TP
    return matches, TP, FP, FN, np.asarray(terr, dtype=float)


def temporal_rpeak_metrics(true_ecg, pred_ecg, fs=8000, tol_ms=50,
                           distance=400, prominence=0.5):
    """Full temporal detection report for ONE segment.

    Returns a dict with detection (Se/PPV/F1/FP/FN), timing-error (ms),
    RR-interval error (ms), and heart-rate error (bpm). Segments with no
    ground-truth peaks return n_true=0 and NaN rates (exclude from means, but
    COUNT them -- a reconstruction that yields peaks where truth has none is a
    real failure; see caller handling).
    """
    tol = int(round(tol_ms / 1000 * fs))
    t_pk = detect_r_peaks(true_ecg, fs, distance, prominence)
    p_pk = detect_r_peaks(pred_ecg, fs, distance, prominence)

    matches, TP, FP, FN, terr = match_peaks(t_pk, p_pk, tol)
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    ppv  = TP / (TP + FP) if (TP + FP) else np.nan
    f1   = (2 * sens * ppv / (sens + ppv)
            if (sens and ppv and not np.isnan(sens) and not np.isnan(ppv)
                and (sens + ppv) > 0) else np.nan)

    terr_ms = np.abs(terr) / fs * 1000.0
    mae_t   = float(np.mean(terr_ms)) if terr_ms.size else np.nan
    med_t   = float(np.median(terr_ms)) if terr_ms.size else np.nan
    iqr_t   = float(np.percentile(terr_ms, 75) - np.percentile(terr_ms, 25)) \
              if terr_ms.size else np.nan

    # RR-interval error on consecutive MATCHED pairs only (avoids missed-beat bias)
    rr_err = np.nan
    if len(matches) >= 2:
        tp_sorted = sorted(matches)
        rr_true, rr_pred = [], []
        for (t0, p0), (t1, p1) in zip(tp_sorted[:-1], tp_sorted[1:]):
            rr_true.append((t1 - t0) / fs * 1000.0)
            rr_pred.append((p1 - p0) / fs * 1000.0)
        rr_err = float(np.mean(np.abs(np.array(rr_true) - np.array(rr_pred))))

    # Heart-rate error (bpm), estimator = 60000 / median-RR
    def _hr(pk):
        if len(pk) < 2:
            return np.nan
        return 60000.0 / np.median(np.diff(pk) / fs * 1000.0)
    hr_true, hr_pred = _hr(t_pk), _hr(p_pk)
    hr_err = float(abs(hr_true - hr_pred)) if not (np.isnan(hr_true) or np.isnan(hr_pred)) else np.nan

    return {
        "n_true": int(len(t_pk)), "n_pred": int(len(p_pk)),
        "TP": TP, "FP": FP, "FN": FN,
        "sensitivity": sens, "ppv": ppv, "f1": f1,
        "mae_timing_ms": mae_t, "median_timing_ms": med_t, "iqr_timing_ms": iqr_t,
        "rr_error_ms": rr_err,
        "hr_true_bpm": hr_true, "hr_pred_bpm": hr_pred, "hr_error_bpm": hr_err,
    }


def aggregate_temporal(rows):
    """Aggregate a list of per-segment temporal_rpeak_metrics dicts.

    Detection totals (TP/FP/FN) are POOLED then Se/PPV/F1 recomputed (the
    correct way to aggregate detection over segments). Timing/RR/HR errors are
    averaged over segments where they are defined; the valid-N is reported so
    the denominator is transparent.
    """
    rows = [r for r in rows if r is not None]
    TP = sum(r["TP"] for r in rows)
    FP = sum(r["FP"] for r in rows)
    FN = sum(r["FN"] for r in rows)
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    ppv  = TP / (TP + FP) if (TP + FP) else np.nan
    f1   = 2 * sens * ppv / (sens + ppv) if (sens + ppv) else np.nan

    def _mean(key):
        v = [r[key] for r in rows if r[key] is not None and not np.isnan(r[key])]
        return (float(np.mean(v)), float(np.std(v)), len(v)) if v else (np.nan, np.nan, 0)

    out = {"n_segments": len(rows), "TP": TP, "FP": FP, "FN": FN,
           "sensitivity": sens, "ppv": ppv, "f1": f1}
    for k in ["mae_timing_ms", "median_timing_ms", "iqr_timing_ms",
              "rr_error_ms", "hr_error_bpm"]:
        m, s, n = _mean(k)
        out[k + "_mean"], out[k + "_std"], out[k + "_validN"] = m, s, n
    return out


# ---------------------------------------------------------------------------
# Preprocessing helpers (shared with the paper pipeline)
# ---------------------------------------------------------------------------
def ensure_positive_polarity(ecg):
    """Flip so R-peaks point up. Applied to the ECG TARGET only (never PCG)."""
    ecg = np.asarray(ecg)
    return -ecg if np.abs(ecg.min()) > np.abs(ecg.max()) else ecg


def normalize_zscore(sig):
    sig = np.asarray(sig, dtype=np.float32)
    return ((sig - sig.mean()) / (sig.std() + 1e-8)).astype(np.float32)

Writing /content/metrics.py


In [6]:
%%writefile /content/ieee_style.py
"""
IEEE double-column figure style for the JBHI revision.

Design goals (per IEEE Transactions / J-BHI figure rules):
  * Vector output (PDF) + high-DPI PNG (600) for every figure.
  * Serif face (Times) to match IEEE body text; DejaVu Serif fallback on Colab.
  * BOLD axis labels that always carry explicit units.
  * Thick spines and ticks so lines stay legible at column scale.
  * constrained_layout everywhere -> no text/label/legend overlaps.
  * Column widths fixed to IEEE spec: single = 3.5 in, double = 7.16 in.

Usage
-----
    from ieee_style import set_ieee_style, new_fig, finalize_axes, save_fig
    set_ieee_style()
    fig, ax = new_fig("single", height=2.2)
    ax.plot(t, y)
    finalize_axes(ax, xlabel="Time (s)", ylabel="Amplitude (a.u.)")
    save_fig(fig, out_dir, "fig_name")   # writes fig_name.pdf AND fig_name.png
"""

import matplotlib
matplotlib.use("Agg")            # headless-safe; remove if you want inline display
import matplotlib.pyplot as plt

# IEEE column widths (inches)
COL_SINGLE = 3.5
COL_DOUBLE = 7.16

# Colour-blind-safe palette (Okabe-Ito), stable role -> colour mapping
COLORS = {
    "truth":    "#009E73",   # bluish green  -> ground truth
    "proposed": "#0072B2",   # blue          -> proposed model
    "cnn":      "#D55E00",   # vermillion    -> CNN
    "unet":     "#CC79A7",   # reddish purple-> U-Net
    "wgan":     "#E69F00",   # orange        -> WGAN-GP
    "pcg":      "#666666",   # grey          -> PCG input
    "accent":   "#0072B2",
    "muted":    "#999999",
}


def set_ieee_style():
    """Apply the global rcParams. Call once at the top of every script."""
    plt.rcParams.update({
        "figure.dpi":        150,
        "savefig.dpi":       600,
        "savefig.bbox":      "tight",
        "savefig.pad_inches": 0.02,
        "figure.constrained_layout.use": True,
        "font.size":         8,
        "font.family":       "serif",
        "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset":  "dejavuserif",
        "text.usetex":       False,
        "axes.labelsize":    9,
        "axes.labelweight":  "bold",
        "axes.titlesize":    9,
        "axes.titleweight":  "bold",
        "axes.linewidth":    1.1,
        "axes.grid":         True,
        "grid.alpha":        0.25,
        "grid.linewidth":    0.4,
        "xtick.labelsize":   8,
        "ytick.labelsize":   8,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.major.size":  3.5,
        "ytick.major.size":  3.5,
        "xtick.direction":   "out",
        "ytick.direction":   "out",
        "legend.fontsize":   7.5,
        "legend.framealpha": 0.92,
        "legend.edgecolor":  "0.7",
        "legend.handlelength": 1.6,
        "lines.linewidth":   1.0,
        "lines.solid_capstyle": "round",
    })


def new_fig(width="single", height=2.2, nrows=1, ncols=1, **kw):
    """Create a figure sized to an IEEE column.

    width : 'single' (3.5 in), 'double' (7.16 in), or a float in inches.
    height: figure height in inches.
    Returns (fig, ax) or (fig, axes) exactly like plt.subplots.
    """
    w = {"single": COL_SINGLE, "double": COL_DOUBLE}.get(width, width)
    fig, ax = plt.subplots(nrows, ncols, figsize=(w, height), **kw)
    return fig, ax


def finalize_axes(ax, xlabel=None, ylabel=None, title=None,
                  legend=False, legend_loc="best", legend_ncol=1):
    """Apply bold labels with units, thicken spines, tidy ticks.

    Always pass xlabel/ylabel WITH units, e.g. 'Time (ms)', 'PSNR (dB)'.
    """
    if xlabel:
        ax.set_xlabel(xlabel, fontweight="bold")
    if ylabel:
        ax.set_ylabel(ylabel, fontweight="bold")
    if title:
        ax.set_title(title, fontweight="bold")
    for spine in ax.spines.values():
        spine.set_linewidth(1.1)
    ax.tick_params(axis="both", which="major", width=1.0, labelsize=8)
    if legend:
        leg = ax.legend(loc=legend_loc, ncol=legend_ncol, frameon=True)
        leg.get_frame().set_linewidth(0.6)
    return ax


def save_fig(fig, out_directory, name):
    """Save both a vector PDF (for LaTeX) and a 600-dpi PNG (for preview)."""
    import os
    pdf = os.path.join(out_directory, f"{name}.pdf")
    png = os.path.join(out_directory, f"{name}.png")
    fig.savefig(pdf)
    fig.savefig(png)
    plt.close(fig)
    print(f"  saved  {pdf}")
    print(f"  saved  {png}")
    return pdf, png

Writing /content/ieee_style.py


In [7]:
import importlib
import config_paths, models, metrics, ieee_style
for m in (config_paths, models, metrics, ieee_style):
    importlib.reload(m)
ieee_style.set_ieee_style()
print('shared modules ready | device =', config_paths.DEVICE)

shared modules ready | device = cuda


## Experiment

# exp7 - Retrain the reconstruction baselines (CNN, U-Net, WGAN-GP)

The original baseline weights were permanently deleted from a university Google
account, so we retrain all three PCG-to-ECG baselines. This notebook then
evaluates all FOUR models (the three retrained baselines + the existing Proposed
generator) on the full 453-segment EPHNOGRAM test set, and produces:
  * the three baseline `best_model.pt` files (so exp2/exp5 can load them),
  * per-segment metrics + paired Wilcoxon vs Proposed (Reviewer 1 comment 2) --
    THE MAIN DELIVERABLE,
  * a subject-clustered bootstrap CI for the Proposed correlation,
  * a REFERENCE table of the retrained baseline scores (for transparency only).

DECISION (per the author): the paper's Table II KEEPS its original numbers
(14.80 / 16.28 / 11.72 ...). The retrained models are used ONLY to compute the
R1.2 per-segment paired statistics; they do NOT replace Table II. The reference
table below is emitted under a distinct label so it can go in the rebuttal or
supplement, not over the main results table.

Faithful to the originals (verified from the source notebooks):
  * SAME architectures (models.py CNNEncoderDecoder / UNetGenerator + WGANCritic),
  * SAME losses (CNN/U-Net plain L1; WGAN-GP: 100*L1 + Wasserstein + 10*GP, n_critic 5),
  * SAME optimizers (Adam; CNN/U-Net betas 0.5/0.999, WGAN betas 0.0/0.9, lr 1e-4),
  * SAME per-segment pipeline (polarity on ECG only -> augment -> z-score both),
  * SAME cosine schedule (T_max 100, eta_min 1e-6) and grad clip 1.0.

Two deliberate CORRECTIONS of latent bugs in the originals (documented, not hidden):
  1. Evaluate the U-Net on all 453 test segments (the paper's row used only 10).
  2. Proper early stopping on the true selection metric (the originals used a
     mode='max' object fed a loss, which halted training early). Best-model
     selection itself was always correct.
  3. Score every model with the shared metrics in the correct (target, prediction)
     order, so all four models are directly comparable (the original WGAN eval
     swapped the arguments).
Because of these corrections + a fresh random seed, the retrained numbers differ
slightly from the paper's originals. That is fine: Table II stays unchanged; the
retrained models only feed the R1.2 paired test, framed as "faithfully retrained
baselines under the same protocol." The Proposed model is NOT retrained.

Runs on a FREE T4 (baselines have no attention -> no OOM). Training is RESUMABLE:
if the T4 session disconnects, just re-run the notebook and each model continues
from its last epoch. To fit short free-tier sessions, train ONE model per session
by editing RUN below (e.g. RUN = ['CNN']).

In [8]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Not on Colab (or already mounted):', exc)
import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

import os, json, time, copy, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.interpolate import interp1d
import scipy.signal as sgn

import config_paths, models, metrics, ieee_style
ieee_style.set_ieee_style()

DEVICE = config_paths.DEVICE
FS, SEG_LEN = config_paths.FS, config_paths.SEG_LEN
OUT = config_paths.out_dir('exp7_baseline_retrain')
print('device:', DEVICE, '| output:', OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda | output: /content/drive/MyDrive/JBHI_Revision/exp7_baseline_retrain


## Protocol (per-model, matched to the originals)

In [9]:
EPOCHS      = 100
ETA_MIN     = 1e-6
GRAD_CLIP   = 1.0
PATIENCE    = 20        # proper early stop on the selection metric
SEED        = 42

# --- GPU / speed configuration (auto-detects Colab Pro A100 / V100 / T4) ---
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True        # TF32 matmul (Ampere+)
torch.backends.cudnn.allow_tf32 = True
_cuda = torch.cuda.is_available()
_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if _cuda else 0.0
USE_AMP   = _cuda and torch.cuda.is_bf16_supported()   # bf16 autocast (no GradScaler needed)
AMP_DTYPE = torch.bfloat16
BATCH_MULT = 2 if _gb >= 35 else 1                     # A100 (40 GB) -> double the batch
NUM_WORKERS = min(8, (os.cpu_count() or 4)) if _cuda else 2
print(f'GPU: {_gb:.0f} GB | bf16 AMP: {USE_AMP} | batch x{BATCH_MULT} | workers: {NUM_WORKERS}')

# NOTE: validation now uses val L1 on the GPU for ALL models (fast); the original
# U-Net selected on val SSIM (skimage, CPU, per sample) which dominated epoch time.
# A unified val-L1 selection is also cleaner for the same-protocol story. SSIM is
# still reported in the final full-453 test evaluation.
BASELINES = {
    'CNN': dict(
        arch='CNNEncoderDecoder', adversarial=False, l1_weight=1.0,
        batch=32 * BATCH_MULT, lr=1e-4, betas=(0.5, 0.999), select='val_loss', mode='min'),
    'UNet': dict(
        arch='UNetGenerator', adversarial=False, l1_weight=1.0,
        batch=32 * BATCH_MULT, lr=1e-4, betas=(0.5, 0.999), select='val_loss', mode='min'),
    'WGAN-GP': dict(
        arch='UNetGenerator', adversarial=True, l1_weight=100.0,
        lambda_gp=10.0, n_critic=5, batch=16 * BATCH_MULT, lr=1e-4, betas=(0.0, 0.9),
        select='val_loss', mode='min'),
}
# Which baselines to (re)train this session. Each skips if its checkpoint exists.
RUN = ['CNN', 'UNet', 'WGAN-GP']


def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def build(arch):
    if arch == 'CNNEncoderDecoder':
        return models.CNNEncoderDecoder(in_channels=1, out_channels=1, base_filters=64)
    if arch == 'UNetGenerator':
        return models.UNetGenerator(in_channels=1, out_channels=1, features=64)
    raise ValueError(arch)


def ckpt_path(name):
    # save where config_paths.CKPT expects it, so exp2/exp5 load it unchanged
    return config_paths.CKPT[name]

GPU: 24 GB | bf16 AMP: True | batch x1 | workers: 8


## Augmentation (train only; identical to the original pipeline)

In [10]:
def _augment(ecg, pcg):
    ecg = ecg.copy(); pcg = pcg.copy()
    if np.random.rand() < 0.5:                        # amplitude scaling
        s = np.random.uniform(0.9, 1.1); ecg *= s; pcg *= s
    if np.random.rand() < 0.6:                        # heart-rate time-stretch
        scale = np.random.uniform(0.85, 1.15)
        n = len(ecg); m = max(2, int(n / scale)); idx = np.linspace(0, n - 1, m)
        ecg = interp1d(np.arange(n), ecg, kind='cubic', fill_value='extrapolate')(idx)
        pcg = interp1d(np.arange(n), pcg, kind='cubic', fill_value='extrapolate')(idx)
        if m >= n:
            ecg, pcg = ecg[:n], pcg[:n]
        else:
            ecg = np.pad(ecg, (0, n - m), mode='edge'); pcg = np.pad(pcg, (0, n - m), mode='edge')
    if np.random.rand() < 0.3:                        # baseline wander
        f = np.random.uniform(0.1, 1.0); t = np.arange(len(ecg)) / FS
        w = 0.1 * np.sin(2 * np.pi * f * t); ecg = ecg + w; pcg = pcg + 0.3 * w
    if np.random.rand() < 0.2:                        # EMG noise, 20-200 Hz
        snr = np.random.uniform(25, 40)
        sos = sgn.butter(4, [20, 200], btype='band', fs=FS, output='sos')
        nz = sgn.sosfilt(sos, np.random.randn(len(ecg)))
        p = np.mean(ecg ** 2) / (10 ** (snr / 10))
        nz = nz * np.sqrt(p / (np.mean(nz ** 2) + 1e-12))
        ecg = ecg + nz; pcg = pcg + 0.5 * nz
    return ecg.astype(np.float32), pcg.astype(np.float32)

## In-RAM dataset (preload once; polarity on ECG only, augment train, z-score both)

In [11]:
_RAM = {}


def _fit(a):
    a = np.asarray(a, np.float32).ravel()
    return a[:SEG_LEN] if a.shape[0] >= SEG_LEN else np.concatenate([a, np.zeros(SEG_LEN - a.shape[0], np.float32)])


def _load_split():
    sp = json.load(open(config_paths.SPLIT_FILE))
    g = lambda *k: next((set(map(str, sp[x])) for x in k if x in sp), set())
    return g('train_subjects', 'train'), g('val_subjects', 'val'), g('test_subjects', 'test')


def _index():
    cache = os.path.join(OUT, 'chunk_index.json')
    if os.path.exists(cache):
        return json.load(open(cache))
    import glob
    idx = {}
    for p in sorted(glob.glob(os.path.join(config_paths.EPH_CHUNKS_DIR, '*.npz'))):
        sid = str(np.load(p, allow_pickle=True)['subject_id'])
        idx.setdefault(sid, []).append(p)
    json.dump(idx, open(cache, 'w'))
    return idx


def _preload(paths, tag):
    if tag in _RAM:
        return _RAM[tag]
    n = len(paths); pcg = np.zeros((n, SEG_LEN), np.float32); ecg = np.zeros((n, SEG_LEN), np.float32)
    t0 = time.time()
    for i, p in enumerate(paths):
        d = np.load(p, allow_pickle=True); pcg[i] = _fit(d['pcg']); ecg[i] = _fit(d['ecg'])
    print(f'  RAM [{tag}]: {n} chunks, {(pcg.nbytes + ecg.nbytes) / 1e9:.2f} GB, {time.time() - t0:.0f}s')
    _RAM[tag] = (pcg, ecg); return _RAM[tag]


class Chunks(Dataset):
    def __init__(self, paths, tag, augment=False):
        self.pcg, self.ecg = _preload(paths, tag); self.augment = augment

    def __len__(self):
        return self.pcg.shape[0]

    def __getitem__(self, i):
        ecg = metrics.ensure_positive_polarity(self.ecg[i])   # ECG only, before aug
        pcg = self.pcg[i].copy()
        if self.augment:
            ecg, pcg = _augment(ecg, pcg)
        ecg = metrics.normalize_zscore(ecg); pcg = metrics.normalize_zscore(pcg)
        return torch.from_numpy(pcg).unsqueeze(0), torch.from_numpy(ecg).unsqueeze(0)


def loaders(batch):
    tr, va, te = _load_split(); idx = _index()
    def paths(ss): return [p for s in sorted(ss) for p in idx.get(s, [])]
    tp, vp, ep = paths(tr), paths(va), paths(te)
    print(f'  train {len(tp)} | val {len(vp)} | test {len(ep)} segments')
    tl = DataLoader(Chunks(tp, 'train', augment=True), batch_size=batch, shuffle=True,
                    drop_last=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'),
                    persistent_workers=(NUM_WORKERS > 0),
                    prefetch_factor=(4 if NUM_WORKERS > 0 else None))
    vl = DataLoader(Chunks(vp, 'val'), batch_size=batch, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))
    return tl, vl, ep

## Validation metric (L1 + SSIM), shared by all models

In [12]:
@torch.no_grad()
def validate(gen, vl):
    """Fast validation: mean L1 on the GPU (no CPU/skimage per-sample loop)."""
    gen.eval(); tot, n = 0.0, 0
    for pcg, ecg in vl:
        pcg = pcg.to(DEVICE, non_blocking=True); ecg = ecg.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
            fake = gen(pcg)
        tot += torch.mean(torch.abs(fake.float() - ecg)).item() * pcg.size(0); n += pcg.size(0)
    return tot / max(n, 1)

## Trainers (regression for CNN/U-Net; WGAN-GP with critic + gradient penalty)

In [13]:
def he_init(m):
    for mod in m.modules():
        if isinstance(mod, (nn.Conv1d, nn.ConvTranspose1d)):
            nn.init.kaiming_normal_(mod.weight, a=0, nonlinearity='relu')
            if mod.bias is not None:
                nn.init.zeros_(mod.bias)


def _better(cur, best, mode):
    return (cur < best) if mode == 'min' else (cur > best)


def train_baseline(name, cfg):
    path = ckpt_path(name)
    done = os.path.join(OUT, f'{name}.done')
    resume = os.path.join(OUT, f'{name}_resume.pt')
    if os.path.exists(done) and os.path.exists(path):
        print(f'[{name}] already trained -> skip.'); return path
    os.makedirs(os.path.dirname(path), exist_ok=True)
    set_seed(SEED)
    tl, vl, _ = loaders(cfg['batch'])
    gen = build(cfg['arch']).to(DEVICE); he_init(gen)
    opt_g = torch.optim.Adam(gen.parameters(), lr=cfg['lr'], betas=cfg['betas'])
    sch_g = torch.optim.lr_scheduler.CosineAnnealingLR(opt_g, T_max=EPOCHS, eta_min=ETA_MIN)
    l1 = nn.L1Loss()
    adversarial = cfg['adversarial']
    critic = opt_c = sch_c = None
    if adversarial:
        critic = models.WGANCritic(in_channels=2, features=64).to(DEVICE)
        opt_c = torch.optim.Adam(critic.parameters(), lr=cfg['lr'], betas=cfg['betas'])
        sch_c = torch.optim.lr_scheduler.CosineAnnealingLR(opt_c, T_max=EPOCHS, eta_min=ETA_MIN)

    best = float('inf') if cfg['mode'] == 'min' else -float('inf')
    best_ep, bad, start = -1, 0, 0
    if os.path.exists(resume):                    # T4 session died? continue.
        st = torch.load(resume, map_location=DEVICE, weights_only=False)
        gen.load_state_dict(st['gen']); opt_g.load_state_dict(st['opt_g']); sch_g.load_state_dict(st['sch_g'])
        if adversarial and st.get('critic') is not None:
            critic.load_state_dict(st['critic']); opt_c.load_state_dict(st['opt_c']); sch_c.load_state_dict(st['sch_c'])
        best, best_ep, bad, start = st['best'], st['best_ep'], st['bad'], st['epoch'] + 1
        print(f'[{name}] resume from epoch {start} (best {best:.4f}@{best_ep}).')
    ac = lambda: torch.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP)
    for ep in range(start, EPOCHS):
        gen.train(); t0 = time.time()
        for pcg, ecg in tl:
            pcg = pcg.to(DEVICE, non_blocking=True); ecg = ecg.to(DEVICE, non_blocking=True)
            if adversarial:
                for _ in range(cfg['n_critic']):
                    with torch.no_grad(), ac():
                        fake = gen(pcg)
                    with ac():
                        c_real = critic(pcg, ecg).mean(); c_fake = critic(pcg, fake).mean()
                    with torch.autocast('cuda', enabled=False):   # GP in fp32: stable double-backward
                        gp = models.compute_gradient_penalty(
                            critic, pcg.float(), ecg.float(), fake.float(), DEVICE)
                    loss_c = (c_fake.float() - c_real.float()) + cfg['lambda_gp'] * gp
                    opt_c.zero_grad(set_to_none=True); loss_c.backward()
                    nn.utils.clip_grad_norm_(critic.parameters(), GRAD_CLIP); opt_c.step()
                with ac():
                    fake = gen(pcg)
                    loss_g = -critic(pcg, fake).mean() + cfg['l1_weight'] * l1(fake, ecg)
            else:
                with ac():
                    fake = gen(pcg)
                    loss_g = cfg['l1_weight'] * l1(fake, ecg)
            opt_g.zero_grad(set_to_none=True); loss_g.backward()
            nn.utils.clip_grad_norm_(gen.parameters(), GRAD_CLIP); opt_g.step()
        sch_g.step()
        if adversarial:
            sch_c.step()

        v_l1 = validate(gen, vl)
        cur = v_l1
        if _better(cur, best, cfg['mode']):
            best, best_ep, bad = cur, ep, 0
            torch.save({'epoch': ep, 'model_state_dict': gen.state_dict(),
                        'model_name': name, 'select': cfg['select'], 'select_value': cur},
                       path)
        else:
            bad += 1
        torch.save({'epoch': ep, 'gen': gen.state_dict(), 'opt_g': opt_g.state_dict(),
                    'sch_g': sch_g.state_dict(),
                    'critic': critic.state_dict() if adversarial else None,
                    'opt_c': opt_c.state_dict() if adversarial else None,
                    'sch_c': sch_c.state_dict() if adversarial else None,
                    'best': best, 'best_ep': best_ep, 'bad': bad}, resume)
        if ep % 2 == 0 or ep == EPOCHS - 1:
            print(f'[{name}] ep {ep:3d}  valL1 {v_l1:.4f}  '
                  f'best {best:.4f}@{best_ep}  ({time.time()-t0:.0f}s/epoch)')
        if bad >= PATIENCE:
            print(f'[{name}] early stop at epoch {ep} (best {best:.4f}@{best_ep}).'); break
    open(done, 'w').write(f'best {best:.6f} @ epoch {best_ep}\n')
    if os.path.exists(resume):
        os.remove(resume)
    print(f'[{name}] done -> {path}')
    return path


for _n in RUN:
    print('=' * 60); print('TRAIN', _n); print('=' * 60)
    try:
        train_baseline(_n, BASELINES[_n])
    except Exception as exc:
        print(f'[{_n}] failed -> {exc}')

TRAIN CNN
  train 4059 | val 681 | test 453 segments
  RAM [train]: 4059 chunks, 2.08 GB, 342s
  RAM [val]: 681 chunks, 0.35 GB, 70s
[CNN] ep   0  valL1 0.6882  best 0.6882@0  (47s/epoch)
[CNN] ep   2  valL1 0.6908  best 0.6606@1  (28s/epoch)
[CNN] ep   4  valL1 0.6690  best 0.6606@1  (28s/epoch)
[CNN] ep   6  valL1 0.6945  best 0.6606@1  (29s/epoch)
[CNN] ep   8  valL1 0.6857  best 0.6606@1  (28s/epoch)
[CNN] ep  10  valL1 0.6810  best 0.6606@1  (29s/epoch)
[CNN] ep  12  valL1 0.6885  best 0.6606@1  (28s/epoch)
[CNN] ep  14  valL1 0.6891  best 0.6606@1  (28s/epoch)
[CNN] ep  16  valL1 0.6737  best 0.6606@1  (29s/epoch)
[CNN] ep  18  valL1 0.7214  best 0.6606@1  (28s/epoch)
[CNN] ep  20  valL1 0.6984  best 0.6606@1  (28s/epoch)
[CNN] early stop at epoch 21 (best 0.6606@1).
[CNN] done -> /content/drive/MyDrive/JBHI_Revision/baseline_checkpoints/CNN_EncoderDecoder/best_model.pt
TRAIN UNet
  train 4059 | val 681 | test 453 segments
[UNet] ep   0  valL1 0.7832  best 0.7832@0  (44s/epoch)
[

## Evaluate ALL FOUR models on the full 453-segment test set
Correct (target, prediction) metric order; no amplitude rescaling (blind).

In [14]:
import pandas as pd


def load_baseline(name):
    if not os.path.exists(ckpt_path(name)):
        return None
    m = build(BASELINES[name]['arch'])
    m, _ = models.load_model(m, ckpt_path(name), device=DEVICE)
    return m


def load_proposed():
    p = config_paths.CKPT['Proposed']
    if not os.path.exists(p):
        p = config_paths.CKPT_PROPOSED_ALT
    m, _ = models.load_model(models.ImprovedGeneratorNoTanh(), p, device=DEVICE)
    return m


_tr, _va, _te = _load_split(); _idx = _index()
test_paths = [p for s in sorted(_te) for p in _idx.get(s, [])]
print('test segments:', len(test_paths))
loaded = {}
for n in ['CNN', 'UNet', 'WGAN-GP']:
    mdl = load_baseline(n)
    if mdl is not None:
        loaded[n] = (mdl, False)
prop = load_proposed()
loaded['Proposed'] = (prop, True)

rows = []
for p in test_paths:
    d = np.load(p, allow_pickle=True)
    ref = metrics.normalize_zscore(metrics.ensure_positive_polarity(_fit(d['ecg'])))
    pcg = metrics.normalize_zscore(_fit(d['pcg']))
    sid = str(d['subject_id'])
    pcg_t = torch.from_numpy(pcg).unsqueeze(0).unsqueeze(0).to(DEVICE)
    scen = torch.zeros(1, config_paths.NUM_SCENARIOS, device=DEVICE)
    scen[0, config_paths.scenario_to_index(d['scenario'])] = 1.0
    for name, (mdl, is_prop) in loaded.items():
        fake = models.run_generator(mdl, pcg_t, scen, is_prop)
        rows.append(dict(chunk=os.path.basename(p), subject=sid, model=name,
                         psnr=metrics.calc_psnr(ref, fake),
                         corr=metrics.calc_correlation(ref, fake),
                         ssim=metrics.calc_ssim(ref, fake),
                         rmse=metrics.calc_rmse(ref, fake),
                         rpeak_amp_err=metrics.calc_r_peak_amplitude_error(
                             ref, fake, fs=FS, distance=config_paths.RPEAK_DISTANCE,
                             prominence=config_paths.RPEAK_PROMINENCE)))
df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT, 'exp7_persegment_metrics.csv'), index=False)
print('per-segment rows:', len(df), '| models:', sorted(df.model.unique()))

test segments: 453
[load_model] best_model.pt: exact match (92 tensors).
[load_model] best_model.pt: exact match (82 tensors).
[load_model] best_model.pt: exact match (82 tensors).
[load_model] trail4_best.pt: exact match (200 tensors).
per-segment rows: 1812 | models: ['CNN', 'Proposed', 'UNet', 'WGAN-GP']


## Aggregate table (updated same-protocol Table II) + paired Wilcoxon vs Proposed

In [15]:
from scipy.stats import wilcoxon

METS = ['psnr', 'corr', 'ssim', 'rpeak_amp_err', 'rmse']
order = [m for m in ['CNN', 'UNet', 'WGAN-GP', 'Proposed'] if m in df.model.unique()]
summary = {}
for m in order:
    sub = df[df.model == m]
    summary[m] = {k: (float(sub[k].mean()), float(sub[k].std())) for k in METS}
print('\n=== Updated Table II (mean +/- SD over 453 segments) ===')
print(f'{"model":<10}' + ''.join(f'{k:>16}' for k in METS))
for m in order:
    print(f'{m:<10}' + ''.join(f'{summary[m][k][0]:>9.3f}+-{summary[m][k][1]:<5.2f}' for k in METS))

# paired Wilcoxon: Proposed vs each baseline, per metric, Holm-Bonferroni
paired = []
if 'Proposed' in order:
    pv = df[df.model == 'Proposed'].set_index('chunk')
    for base in [m for m in order if m != 'Proposed']:
        bv = df[df.model == base].set_index('chunk')
        common = pv.index.intersection(bv.index)
        for k in METS:
            a = pv.loc[common, k].values; b = bv.loc[common, k].values
            good = ~(np.isnan(a) | np.isnan(b))
            try:
                stat, p = wilcoxon(a[good], b[good])
            except Exception:
                stat, p = np.nan, np.nan
            paired.append(dict(baseline=base, metric=k, statistic=float(stat), p_raw=float(p)))
    # Holm-Bonferroni across the whole family
    ps = sorted([(r['p_raw'], i) for i, r in enumerate(paired) if not np.isnan(r['p_raw'])])
    mtot = len(ps)
    for rank, (p, i) in enumerate(ps):
        paired[i]['p_holm'] = float(min(1.0, p * (mtot - rank)))
pd.DataFrame(paired).to_csv(os.path.join(OUT, 'exp7_paired_wilcoxon.csv'), index=False)

# subject-clustered bootstrap CI for the Proposed correlation (effective n=3)
prop_corr = df[df.model == 'Proposed']
subj = sorted(prop_corr.subject.unique())
means = {s: prop_corr[prop_corr.subject == s]['corr'].mean() for s in subj}
boot = []
rng = np.random.RandomState(SEED)
for _ in range(10000):
    pick = rng.choice(subj, size=len(subj), replace=True)
    boot.append(np.mean([means[s] for s in pick]))
ci = (float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5)))
seg = prop_corr['corr'].values
seg_ci = (float(np.percentile([np.mean(rng.choice(seg, len(seg))) for _ in range(2000)], 2.5)),
          float(np.percentile([np.mean(rng.choice(seg, len(seg))) for _ in range(2000)], 97.5)))
print(f'\nProposed corr: pooled {prop_corr["corr"].mean():.3f} | '
      f'segment-level CI {seg_ci} | subject-clustered CI {ci} (effective n={len(subj)})')

json.dump({'summary': summary, 'paired_wilcoxon': paired,
           'proposed_corr_subject_ci': ci, 'proposed_corr_segment_ci': seg_ci,
           'per_subject_proposed_corr': means, 'n_test': int(len(test_paths))},
          open(os.path.join(OUT, 'exp7_results.json'), 'w'), indent=2)


=== Updated Table II (mean +/- SD over 453 segments) ===
model                 psnr            corr            ssim   rpeak_amp_err            rmse
CNN          12.802+-2.60     0.241+-0.20     0.682+-0.08     2.036+-0.53     1.047+-0.29 
UNet         15.211+-3.79     0.491+-0.32     0.786+-0.11     1.269+-0.62     0.816+-0.24 
WGAN-GP      14.084+-2.73     0.385+-0.28     0.690+-0.14     1.673+-0.67     0.901+-0.17 
Proposed     17.097+-5.07     0.610+-0.40     0.903+-0.07     0.995+-0.66     0.695+-0.32 

Proposed corr: pooled 0.610 | segment-level CI (0.57260367494831, 0.6470126203095816) | subject-clustered CI (0.3135612189593828, 0.9068139779567718) (effective n=3)


## Emit updated Table II (LaTeX) + IEEE figure

In [16]:
def emit_table():
    hdr = {'psnr': 'PSNR (dB)', 'corr': 'Corr.', 'ssim': 'SSIM',
           'rpeak_amp_err': 'R-peak amp. err.', 'rmse': 'RMSE'}
    higher = {'psnr', 'corr', 'ssim'}
    best = {}
    for k in METS:
        vals = {m: summary[m][k][0] for m in order}
        best[k] = (max if k in higher else min)(vals, key=vals.get)
    L = ['% Auto-generated by exp7. REFERENCE ONLY -- retrained baselines used for the',
         '% R1.2 paired test. The paper Table II KEEPS its original numbers; do NOT',
         '% replace it with this. Requires booktabs.',
         '\\begin{table}[t]\\centering', '\\footnotesize',
         '\\caption{Reference: faithfully retrained baselines under a single protocol, '
         'scored on all 453 EPHNOGRAM test segments. Used only for the paired '
         'significance test in the R1.2 response; the main results table is '
         'unchanged. Best in \\textbf{bold}.}',
         '\\label{tab:retrained_ref}', '\\begin{tabular}{l' + 'c' * len(order) + '}', '\\toprule',
         'Metric & ' + ' & '.join(order) + ' \\\\', '\\midrule']
    for k in METS:
        cells = []
        for m in order:
            v = summary[m][k][0]
            s = f'{v:.2f}' if k in ('psnr',) else f'{v:.3f}'
            cells.append(f'\\textbf{{{s}}}' if m == best[k] else s)
        L.append(f'{hdr[k]} & ' + ' & '.join(cells) + ' \\\\')
    L += ['\\bottomrule', '\\end{tabular}', '\\end{table}', '']
    open(os.path.join(OUT, 'exp7_reference_table.tex'), 'w').write('\n'.join(L))
    print('wrote exp7_reference_table.tex (reference only; paper Table II unchanged)')


emit_table()

fig, axes = ieee_style.new_fig('double', height=2.4, ncols=3)
for ax, k, lab in zip(axes, ['psnr', 'corr', 'rpeak_amp_err'],
                      ['PSNR (dB)', 'Pearson r', 'R-peak amp. err. (a.u.)']):
    data = [df[df.model == m][k].dropna().values for m in order]
    bp = ax.boxplot(data, labels=order, showfliers=False, patch_artist=True)
    for patch, c in zip(bp['boxes'], [ieee_style.COLORS[x] for x in ['cnn', 'unet', 'wgan', 'proposed'][:len(order)]]):
        patch.set_facecolor(c); patch.set_alpha(0.6)
    ax.set_xticklabels(order, rotation=25, ha='right', fontsize=7)
    ieee_style.finalize_axes(ax, ylabel=lab)
ieee_style.save_fig(fig, OUT, 'fig_exp7_model_boxplots')
print('\nDONE. Baseline weights saved to the config_paths.CKPT locations; exp2/exp5 '
      'now load them. The paper Table II stays as-is. Use exp7_paired_wilcoxon.csv '
      'for the R1.2 significance test; exp7_reference_table.tex is a rebuttal-only reference.')

wrote exp7_reference_table.tex (reference only; paper Table II unchanged)


/tmp/ipykernel_7244/2924394573.py:37: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=order, showfliers=False, patch_artist=True)
/tmp/ipykernel_7244/2924394573.py:37: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=order, showfliers=False, patch_artist=True)
/tmp/ipykernel_7244/2924394573.py:37: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=order, showfliers=False, patch_artist=True)


  saved  /content/drive/MyDrive/JBHI_Revision/exp7_baseline_retrain/fig_exp7_model_boxplots.pdf
  saved  /content/drive/MyDrive/JBHI_Revision/exp7_baseline_retrain/fig_exp7_model_boxplots.png

DONE. Baseline weights saved to the config_paths.CKPT locations; exp2/exp5 now load them. The paper Table II stays as-is. Use exp7_paired_wilcoxon.csv for the R1.2 significance test; exp7_reference_table.tex is a rebuttal-only reference.
